# Autograd - Build
This is the build section of the project, where new functions, classes, etc are created and tested. Finished peices are moved into main where they are then called and used by build.

### Rules:
- No ai used to generate any code, only for reaserch, explinations and code reviews
- Finished peices must be abstract and state clearly if any cases are not covered

## Setup

In [8]:
import numpy as np
import main
from typing import Union, List
from matplotlib import pyplot as plt
import matplotlib_inline
%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


## Build

In [9]:
class Model():
    """
    
    """
    def __init__(self,
                  input_size : int, hidden_size : int, output_size : int,
                  number_of_layers : int, activation_function, normalisation_function,
                  precision: str = 'float32', random_seed: int = None):
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.number_of_layers = number_of_layers

        self.precision = precision
        self.random_seed = random_seed

        self.activation_function = activation_function
        self.normalisation_function = normalisation_function

        self.activation_function_derivative = main.derivatives.get(
            getattr(self.activation_function, "__name__", None)
        )
        self.normalisation_function_derivative = main.derivatives.get(
            getattr(self.normalisation_function, "__name__", None)
        )


        layers = []
        if self.number_of_layers == 1:
            layers.append(main.LinearLayer(self.input_size, self.output_size,
                                             self.precision, self.random_seed))
        else:
            layers.append(
                main.LinearLayer(self.input_size, self.hidden_size, self.precision, self.random_seed))
            for _ in range(max(0, self.number_of_layers - 2)):
                layers.append(self.activation_function)
                layers.append(main.LinearLayer(self.hidden_size, self.hidden_size, 
                                               self.precision, self.random_seed))
            layers.append(main.LinearLayer(self.hidden_size, self.output_size, 
                                           self.precision, self.random_seed))

        layers.append(self.activation_function)
        layers.append(self.normalisation_function)

        self.modules = layers

        parramaters = {}
        i = 1
        for obj in self.modules:
            if isinstance(obj, main.LinearLayer):
                parramaters[f"Layer {i}"] = obj.parramaters
                i += 1

        self.parramaters = parramaters

    def forward(self, x: Union[np.ndarray, List, float, int]) -> np.ndarray:
        for module in self.modules:
            if hasattr(module, "forward") and callable(module.forward):
                x = module.forward(x)
            elif callable(module):
                x = module(x)
            else:
                raise TypeError(f"Module - {type(module).__name__} does not have a 'forward' function, or is not callable and so is not supported")
        return x


    def set_parramaters(self, parramaters : dict):
        """
        Sets the model parameters to the ones stored in the input dictionary.

        Args:
            parramaters (dict): The dictionary storing the parramaters.

        Updates:
            self.parramaters: Sets self.parramaters["Layer i"] to parramaters["Layer i"]

        Rasies:
            TypeError: If the input is not a dictionary.
            KeyError: If the inputed dictionary does not have the keys "Layer 1" thorugh "Layer i".
        """

        if not isinstance(parramaters, dict):
            raise TypeError(f"Input - {parramaters} - must be a dict.")

        if set(parramaters.keys()) != set(self.parramaters.keys()):
            raise KeyError(f"Input - {parramaters} - keys do not match those in self.parramaters.")

        if len(self.parramaters) != len(parramaters):
            raise ValueError(f"Inputed dictionary has a different number of elements than the existing dictionary.")

        
        modules_copy = self.modules
        i = 1
        for obj in modules_copy:
            if isinstance(obj, main.LinearLayer):
                try:
                    obj.set_parramaters(parramaters[f"Layer {i}"])
                    i += 1
                except (TypeError, ValueError, KeyError) as exc:
                    raise ValueError(f"Could not set layer {i}'s parramaters to those in the inputed dictionary.")

        self.modules = modules_copy
        self.parramaters = parramaters
    

In [10]:
test_inputs = [
    np.array([-1, -0.5, 0]),
    [[0, 1, 2], [2, 0.7, -1]],
    (-2, -0.32, 0),
    "This is a string."
]

model = Model(input_size=3, output_size=2, hidden_size=1,
              number_of_layers=3, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=1)

In [11]:
saved = model.parramaters
print(model.parramaters)
model = Model(input_size=3, output_size=2, hidden_size=1,
              number_of_layers=3, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=4)
print(model.parramaters)
model.set_parramaters(saved)
print(model.parramaters)

{'Layer 1': {'weights': array([[-0.054],
       [ 0.023],
       [ 0.51 ]], dtype=float32), 'biases': array([0.9], dtype=float32)}, 'Layer 2': {'weights': array([[-0.054]], dtype=float32), 'biases': array([0.023], dtype=float32)}, 'Layer 3': {'weights': array([[-0.054,  0.023]], dtype=float32), 'biases': array([0.51, 0.9 ], dtype=float32)}}
{'Layer 1': {'weights': array([[0.452],
       [0.886],
       [0.762]], dtype=float32), 'biases': array([0.022], dtype=float32)}, 'Layer 2': {'weights': array([[0.452]], dtype=float32), 'biases': array([0.886], dtype=float32)}, 'Layer 3': {'weights': array([[0.452, 0.886]], dtype=float32), 'biases': array([0.762, 0.022], dtype=float32)}}
{'Layer 1': {'weights': array([[-0.054],
       [ 0.023],
       [ 0.51 ]], dtype=float32), 'biases': array([0.9], dtype=float32)}, 'Layer 2': {'weights': array([[-0.054]], dtype=float32), 'biases': array([0.023], dtype=float32)}, 'Layer 3': {'weights': array([[-0.054,  0.023]], dtype=float32), 'biases': array([0.5

In [12]:
for t in test_inputs:
    try:
        print(f"Input - {t} - Output - {model.forward(t)}")
        print()
    except:
        print(f"Input - {t} - can not be passed through the model.")

Input - [-1.  -0.5  0. ] - Output - [[0.40423448 0.59576552]]

Input - [[0, 1, 2], [2, 0.7, -1]] - Output - [[0.40523675 0.59476325]
 [0.40358935 0.59641065]]

Input - (-2, -0.32, 0) - Output - [[0.40429269 0.59570731]]

Input - This is a string. - can not be passed through the model.
